# INF632 Homework 3 – Statistical Learning & Prediction

**Student:** Hari Krishna  
**Course:** EE599  

This notebook implements `kmeans(X, k)`, `knn(training_set, k, new_point)`, and `cpa(time_series)` from scratch and applies them to the class datasets.


## 1. K-means validation

Daily steps and daily calories are used together as a two-dimensional dataset. The method remains single-pass, but the centroid initialization uses spread-out observations so the result is less sensitive to row order.

| Cluster | Count | Mean steps | Mean calories |
|---|---:|---:|---:|
| 0 | 284 | 92.32 | 2697.99 |
| 1 | 359 | 5884.43 | 2693.82 |
| 2 | 24 | 14119.46 | 3232.42 |

**Plot explanation.** The K-means scatter plot shows how the observations separate into low-, medium-, and high-activity groups. The centroid markers show the average step-calorie profile of each cluster after one update step.


## 2. KNN validation

Calories are converted into `HighCalorie` and `LowCalorie` classes using the median calorie value, and daily steps are used as the predictor. A holdout accuracy is also reported so the validation is not based on a single example.

**Plot explanation.** The KNN plot places the new point at 8000 steps against the labeled training rows. This makes it easier to see why the local-neighbor decision favors the higher-calorie class for this point.


In [2]:
import csv
import math
from datetime import datetime
from pathlib import Path
import matplotlib.pyplot as plt


def get_base_dir():
    # handle both notebook and script execution so file lookup works in either case
    if "__file__" in globals():
        return Path(__file__).resolve().parent
    return Path.cwd()


def find_data_file(candidates):
    # search common locations first so the submission runs without manual path edits
    roots = []
    base_dir = get_base_dir()
    cwd = Path.cwd()

    roots.append(cwd)
    roots.extend(cwd.parents)
    roots.append(base_dir)
    roots.extend(base_dir.parents)
    roots.append(Path("/mnt/data"))

    seen = set()
    for root in roots:
        root = root.resolve()
        if root in seen:
            continue
        seen.add(root)

        for name in candidates:
            path = root / name
            if path.exists():
                return path

    raise FileNotFoundError(f"Could not find any of these files: {candidates}")


def read_csv_as_dicts(path):
    # read each row as a dictionary so the column mapping stays clear
    rows = []
    with open(path, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append(row)
    return rows


def merge_daily_steps_and_calories(steps_path, calories_path):
    # merge both daily files by date so each row has matching steps and calories
    step_rows = read_csv_as_dicts(steps_path)
    calorie_rows = read_csv_as_dicts(calories_path)

    calories_by_day = {}
    for row in calorie_rows:
        calories_by_day[row["ActivityDay"]] = int(row["Calories"])

    merged = []
    for row in step_rows:
        day = row["ActivityDay"]
        if day in calories_by_day:
            merged.append(
                {
                    "ActivityDay": day,
                    "date": datetime.strptime(day, "%m/%d/%Y"),
                    "StepTotal": int(row["StepTotal"]),
                    "Calories": calories_by_day[day],
                }
            )

    merged.sort(key=lambda row: row["date"])
    return merged


def write_csv(path, rows, fieldnames):
    # save summary tables so the numeric results are easy to inspect outside the notebook
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


def euclidean_distance(p1, p2):
    # compute euclidean distance directly instead of using a library function
    if len(p1) != len(p2):
        raise ValueError("Points must have the same number of dimensions.")

    total = 0.0
    for a, b in zip(p1, p2):
        total += (a - b) ** 2
    return math.sqrt(total)


def mean_of_points(points):
    # take the average of the assigned points to update each centroid
    if len(points) == 0:
        raise ValueError("Cannot compute the mean of an empty cluster.")

    dims = len(points[0])
    centroid = []
    for d in range(dims):
        coordinate_sum = 0.0
        for point in points:
            coordinate_sum += point[d]
        centroid.append(coordinate_sum / len(points))
    return centroid


def median_of_numbers(values):
    # use the median calories value to split the data into two classes for knn
    ordered = sorted(values)
    n = len(ordered)
    mid = n // 2
    if n % 2 == 1:
        return ordered[mid]
    return (ordered[mid - 1] + ordered[mid]) / 2.0


def initialize_manual_centroids(X, k):
    # pick spread-out starting points so the single-pass result is less affected by row order
    ordered = sorted(X, key=lambda row: row[0])

    # return the middle observation directly when only one cluster is requested
    if k == 1:
        return [ordered[len(ordered) // 2][:]]

    centroids = []
    last_index = len(ordered) - 1
    for i in range(k):
        index = round(i * last_index / (k - 1))
        centroids.append(ordered[index][:])
    return centroids


def kmeans(X, k):

    if k <= 0:
        raise ValueError("k must be positive.")
    if len(X) == 0:
        raise ValueError("X must not be empty.")

    # cap k at dataset size so the method still runs even if the requested k is too large
    if k > len(X):
        k = len(X)

    dims = len(X[0])
    for row in X:
        if len(row) != dims:
            raise ValueError("All rows in X must have the same number of columns.")

    centroids = initialize_manual_centroids(X, k)
    clusters = [[] for _ in range(k)]
    assignments = []

    for point in X:
        best_cluster = None
        best_distance = None
        for cluster_id, centroid in enumerate(centroids):
            distance = euclidean_distance(point, centroid)
            if best_distance is None or distance < best_distance:
                best_distance = distance
                best_cluster = cluster_id

        assignments.append(best_cluster)
        clusters[best_cluster].append(point)

    updated_centroids = []
    for cluster_id in range(k):
        # keep the previous centroid when a cluster becomes empty to avoid breaking the update step
        if len(clusters[cluster_id]) == 0:
            updated_centroids.append(centroids[cluster_id][:])
        else:
            updated_centroids.append(mean_of_points(clusters[cluster_id]))

    return {
        "clusters": clusters,
        "centroids": updated_centroids,
        "assignments": assignments,
    }


def knn(training_set, k, new_point):
    # keep the class label in the last column
    if len(training_set) == 0:
        raise ValueError("Training set must not be empty.")

    feature_count = len(training_set[0]) - 1
    if feature_count <= 0:
        raise ValueError("Each training row must include features and a class label.")
    if len(new_point) != feature_count:
        raise ValueError("new_point must match the feature dimension of the training set.")

    # reduce k when the request is larger than the dataset so the calculation still stays valid
    if k > len(training_set):
        k = len(training_set)

    # shift even k down by one so majority voting still produces a single winner
    if k % 2 == 0:
        k -= 1
    if k <= 0:
        raise ValueError("k must be at least 1 after adjustment.")

    neighbors = []
    for row in training_set:
        features = row[:-1]
        label = row[-1]
        distance = euclidean_distance(features, new_point)
        neighbors.append((distance, label, features))

    neighbors.sort(key=lambda item: item[0])
    nearest = neighbors[:k]

    votes = {}
    for _, label, _ in nearest:
        votes[label] = votes.get(label, 0) + 1

    predicted_label = None
    highest_vote_count = -1
    for label, vote_count in votes.items():
        if vote_count > highest_vote_count:
            highest_vote_count = vote_count
            predicted_label = label

    return predicted_label, nearest, k


def segment_sse(values):
    # use within-segment squared error to measure how stable each segment is
    if len(values) == 0:
        return 0.0

    mean_value = sum(values) / len(values)
    sse = 0.0
    for value in values:
        sse += (value - mean_value) ** 2
    return sse


def cpa(time_series, min_segment_size=7):
    # find the strongest single change point by checking which split reduces error the most
    n = len(time_series)
    if n < 2 * min_segment_size:
        raise ValueError("Time series is too short for the chosen minimum segment size.")

    total_sse = segment_sse(time_series)
    best_index = None
    best_gain = None

    for split_index in range(min_segment_size, n - min_segment_size + 1):
        left = time_series[:split_index]
        right = time_series[split_index:]

        split_sse = segment_sse(left) + segment_sse(right)
        gain = total_sse - split_sse

        if best_gain is None or gain > best_gain:
            best_gain = gain
            best_index = split_index

    return best_index, best_gain


def find_multiple_change_points(time_series, max_points=8, min_segment_size=14, min_gain_ratio=0.01):
    # keep splitting the segment with the largest gain so the main change points show up first
    segments = [(0, len(time_series))]
    change_points = []
    total_sse = segment_sse(time_series)
    min_gain = total_sse * min_gain_ratio

    for _ in range(max_points):
        best_candidate = None

        for start, end in segments:
            segment = time_series[start:end]

            # skip segments that are too short to support a meaningful split
            if len(segment) < 2 * min_segment_size:
                continue

            local_cp, gain = cpa(segment, min_segment_size=min_segment_size)
            global_cp = start + local_cp
            candidate = (gain, global_cp, start, end)

            if best_candidate is None or candidate[0] > best_candidate[0]:
                best_candidate = candidate

        # stop early when the remaining gain is too small to represent a meaningful shift
        if best_candidate is None or best_candidate[0] < min_gain:
            break

        gain, cp_index, start, end = best_candidate
        change_points.append((cp_index, gain))

        segments.remove((start, end))
        segments.append((start, cp_index))
        segments.append((cp_index, end))
        segments.sort()

    change_points.sort(key=lambda item: item[0])
    return change_points


def summarize_clusters(X, assignments, k):
    # summarize each cluster with counts and means so the result is easier to explain
    rows = []
    for cluster_id in range(k):
        points = [X[i] for i in range(len(X)) if assignments[i] == cluster_id]
        avg_steps = sum(point[0] for point in points) / len(points)
        avg_calories = sum(point[1] for point in points) / len(points)
        rows.append(
            {
                "cluster": cluster_id,
                "count": len(points),
                "mean_steps": round(avg_steps, 2),
                "mean_calories": round(avg_calories, 2),
            }
        )
    return rows


def summarize_change_points(nonzero_dates, nonzero_steps, change_points):
    # compare the mean steps before and after each breakpoint so the shift is measurable
    boundaries = [0] + [cp for cp, _ in change_points] + [len(nonzero_steps)]
    rows = []

    for i, (cp_index, gain) in enumerate(change_points):
        left_start = boundaries[i]
        left_end = cp_index
        right_start = cp_index
        right_end = boundaries[i + 2]

        left_steps = nonzero_steps[left_start:left_end]
        right_steps = nonzero_steps[right_start:right_end]

        left_mean = sum(left_steps) / len(left_steps)
        right_mean = sum(right_steps) / len(right_steps)
        delta = right_mean - left_mean

        if delta > 0:
            interpretation = "Activity increases after this date."
        else:
            interpretation = "Activity decreases after this date."

        rows.append(
            {
                "change_point_index": cp_index,
                "date": nonzero_dates[cp_index].strftime("%Y-%m-%d"),
                "gain": round(gain, 2),
                "mean_before": round(left_mean, 2),
                "mean_after": round(right_mean, 2),
                "difference_after_minus_before": round(delta, 2),
                "interpretation": interpretation,
            }
        )

    return rows


def make_kmeans_plot(X, assignments, centroids, out_path):
    # plot steps against calories to show how the clusters separate in feature space
    plt.figure(figsize=(8, 5))
    for cluster_id in sorted(set(assignments)):
        xs = [X[i][0] for i in range(len(X)) if assignments[i] == cluster_id]
        ys = [X[i][1] for i in range(len(X)) if assignments[i] == cluster_id]
        plt.scatter(xs, ys, label=f"Cluster {cluster_id}")

    centroid_x = [c[0] for c in centroids]
    centroid_y = [c[1] for c in centroids]
    plt.scatter(centroid_x, centroid_y, marker="x", s=140, linewidths=2, label="Centroids")
    plt.xlabel("Daily steps")
    plt.ylabel("Daily calories")
    plt.title("K-means clustering on daily steps and calories")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.close()


def make_knn_plot(training_set, new_point, out_path):
    # plot the labeled step values and the new point to show the knn decision visually
    plt.figure(figsize=(8, 5))
    low_x = [row[0] for row in training_set if row[-1] == "LowCalorie"]
    high_x = [row[0] for row in training_set if row[-1] == "HighCalorie"]

    plt.scatter(low_x, [0] * len(low_x), label="LowCalorie")
    plt.scatter(high_x, [1] * len(high_x), label="HighCalorie")
    plt.scatter([new_point[0]], [0.5], marker="x", s=140, linewidths=2, label="New point")

    plt.yticks([0, 1], ["LowCalorie", "HighCalorie"])
    plt.xlabel("Daily steps")
    plt.ylabel("Calorie class")
    plt.title("KNN validation using daily steps")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.close()


def make_cpa_plot(nonzero_dates, nonzero_steps, change_points, out_path):
    # plot the nonzero step series and mark the detected change points
    plt.figure(figsize=(11, 5))
    plt.plot(nonzero_dates, nonzero_steps, label="Daily steps")

    first_line = True
    for cp_index, _ in change_points:
        plt.axvline(
            nonzero_dates[cp_index],
            linestyle="--",
            label="Change point" if first_line else None,
        )
        first_line = False

    plt.xlabel("Date")
    plt.ylabel("Step count")
    plt.title("Nonzero daily steps with detected change points")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.close()


if __name__ == "__main__":
    steps_file = find_data_file(["dailySteps (1).csv", "dailySteps.csv"])
    calories_file = find_data_file(["dailyCalories.csv", "dailyCalories (1).csv"])

    merged = merge_daily_steps_and_calories(steps_file, calories_file)

    print("Using files:", steps_file.name, "and", calories_file.name)
    print("Merged daily records:", len(merged))

    X = [[row["StepTotal"], row["Calories"]] for row in merged]
    requested_kmeans_k = 3
    km = kmeans(X, k=requested_kmeans_k)
    cluster_summary = summarize_clusters(X, km["assignments"], len(km["centroids"]))
    write_csv("cluster_summary.csv", cluster_summary, ["cluster", "count", "mean_steps", "mean_calories"])

    print("\nK-MEANS RESULTS")
    print("Requested k:", requested_kmeans_k)
    print("Used k:", len(km["centroids"]))
    print("Centroids [steps, calories]:")
    for i, centroid in enumerate(km["centroids"]):
        print(f"  Cluster {i}: {[round(centroid[0], 2), round(centroid[1], 2)]}")
    print("Cluster summary:")
    for row in cluster_summary:
        print(row)

    make_kmeans_plot(X, km["assignments"], km["centroids"], "plot_kmeans_steps_calories.png")

    calorie_median = median_of_numbers([row["Calories"] for row in merged])
    training_set = []
    for row in merged[:100]:
        label = "HighCalorie" if row["Calories"] >= calorie_median else "LowCalorie"
        training_set.append([row["StepTotal"], label])

    requested_knn_k = 5
    new_point = [8000]
    knn_label, neighbors, used_knn_k = knn(training_set, k=requested_knn_k, new_point=new_point)

    correct = 0
    test_rows = []
    for row in merged[100:160]:
        true_label = "HighCalorie" if row["Calories"] >= calorie_median else "LowCalorie"
        predicted_label, _, _ = knn(training_set, k=requested_knn_k, new_point=[row["StepTotal"]])
        if predicted_label == true_label:
            correct += 1
        test_rows.append((row["StepTotal"], true_label, predicted_label))
    knn_accuracy = correct / len(test_rows)

    print("\nKNN RESULT")
    print("Requested k:", requested_knn_k)
    print("Used k:", used_knn_k)
    print("New point:", new_point)
    print("Predicted label:", knn_label)
    print("Holdout accuracy:", round(knn_accuracy, 4))
    print("Nearest neighbors:")
    for distance, label, features in neighbors:
        print(f"  steps={features[0]}, label={label}, distance={round(distance, 2)}")

    make_knn_plot(training_set, new_point, "plot_knn_steps_class.png")

    nonzero_dates = []
    nonzero_steps = []
    for row in merged:
        if row["StepTotal"] != 0:
            nonzero_dates.append(row["date"])
            nonzero_steps.append(row["StepTotal"])

    change_points = find_multiple_change_points(
        nonzero_steps,
        max_points=8,
        min_segment_size=14,
        min_gain_ratio=0.01
    )
    change_summary = summarize_change_points(nonzero_dates, nonzero_steps, change_points)
    write_csv(
        "change_point_summary.csv",
        change_summary,
        [
            "change_point_index",
            "date",
            "gain",
            "mean_before",
            "mean_after",
            "difference_after_minus_before",
            "interpretation",
        ],
    )

    print("\nCHANGE POINT RESULTS")
    print("Detected points:", len(change_points))
    for row in change_summary:
        print(row)

    make_cpa_plot(nonzero_dates, nonzero_steps, change_points, "plot_change_points_steps.png")

    print("\nSHORT ANALYSIS")
    print("K-means separates the observations into low-, medium-, and high-activity groups.")
    print("KNN gives a reasonable calorie-class prediction, and the holdout check supports that the rule is stable.")
    print("The change-point table shows that the detected dates align with clear shifts in average activity level.")
    print("Several summer breakpoints support a seasonal pattern rather than random day-to-day noise.")

Using files: dailySteps (1).csv and dailyCalories.csv
Merged daily records: 667

K-MEANS RESULTS
Requested k: 3
Used k: 3
Centroids [steps, calories]:
  Cluster 0: [92.32, 2697.99]
  Cluster 1: [5884.43, 2693.82]
  Cluster 2: [14119.46, 3232.42]
Cluster summary:
{'cluster': 0, 'count': 284, 'mean_steps': 92.32, 'mean_calories': 2697.99}
{'cluster': 1, 'count': 359, 'mean_steps': 5884.43, 'mean_calories': 2693.82}
{'cluster': 2, 'count': 24, 'mean_steps': 14119.46, 'mean_calories': 3232.42}

KNN RESULT
Requested k: 5
Used k: 5
New point: [8000]
Predicted label: HighCalorie
Holdout accuracy: 0.8
Nearest neighbors:
  steps=8110, label=HighCalorie, distance=110.0
  steps=8200, label=HighCalorie, distance=200.0
  steps=7709, label=HighCalorie, distance=291.0
  steps=7685, label=HighCalorie, distance=315.0
  steps=7681, label=HighCalorie, distance=319.0

CHANGE POINT RESULTS
Detected points: 5
{'change_point_index': 198, 'date': '2013-07-15', 'gain': 121525893.16, 'mean_before': 5981.57, 'me

## 3. Change-point analysis on nonzero daily steps

Zero-step days are removed. The dominant single-change-point rule is then applied repeatedly, and the mean activity level before and after each detected date is summarized.

| Date | Mean before | Mean after | Difference | Interpretation |
|---|---:|---:|---:|---|
| 2013-07-15 | 5981.57 | 2932.93 | -3048.64 | Activity decreases after this date. |
| 2013-07-29 | 2932.93 | 8522.84 | 5589.91 | Activity increases after this date. |
| 2013-08-18 | 8522.84 | 4270.26 | -4252.58 | Activity decreases after this date. |
| 2014-06-25 | 4270.26 | 191.26 | -4079.0 | Activity decreases after this date. |
| 2014-08-29 | 191.26 | 5571.51 | 5380.25 | Activity increases after this date. |

**Plot explanation.** The time-series plot shows the nonzero daily steps across the full study period. The dashed vertical lines mark the dates where the average level shifts most strongly, and the summary table quantifies the size and direction of each shift.


## 4. Short analysis

The K-means centroids and cluster summary support three interpretable groups: low, moderate, and high activity. That structure matches the expected relationship between movement and calorie expenditure.

The KNN validation is also reasonable because the nearest neighbors to 8000 steps all fall in the higher-calorie class, and the holdout accuracy suggests that the rule is fairly stable.

For change-point analysis, the summary table shows that the detected dates correspond to clear shifts in mean activity level rather than isolated outliers. The largest positive shifts occur around late July 2013 and late August 2014, while the strongest decreases occur around mid-August 2013 and late June 2014. Several changes occur in June, July, and August, which supports a seasonal pattern tied to summer routines, travel, weather, or schedule changes.
